In [ ]:
from nba_api.stats.endpoints import leaguegamelog

# Initializing with the exact parameters expected by the underlying wrapper class
logs = leaguegamelog.LeagueGameLog(
    season="2024-25",
    season_type_all_star="Regular Season",
    player_or_team_abbreviation="T"  # 'T' forces team logs instead of player rows
)

df = logs.get_data_frames()[0]


In [ ]:
import pandas as pd
from sklearn.linear_model import LinearRegression
from matplotlib import pyplot as plt
df['OWN_TEAM_MISSES'] = (df['FGA'] - df['FGM']) + ((df['FTA'] - df['FTM']) * 0.44)
Team_lookup = df[['GAME_ID', 'TEAM_ID', 'OWN_TEAM_MISSES']]
merged_df = pd.merge(df, 
    Team_lookup, 
    on='GAME_ID', 
    suffixes=('', '_team') # Automatically appends '_opp' to the duplicate columns from Table B
)
final_df = merged_df[merged_df['TEAM_ID'] != merged_df['TEAM_ID_opp']]


opp_misses_reb = final_df[["TEAM_MISSES_opp", "REB"]]
opp_misses_reb.head()
opp_misses_reb.shape
opp_misses_reb.isnull().sum()
print(df.shape)
print(final_df.shape)

plt.scatter(opp_misses_reb["TEAM_MISSES_opp"], opp_misses_reb["REB"])
plt.xlim(0, 100)
plt.ylim(0, 100)
plt.xlabel("Opp Team Misses")
plt.ylabel("Rebound")
plt.show()
print("Correlation:", opp_misses_reb["TEAM_MISSES_opp"].corr(opp_misses_reb["REB"]))

X= opp_misses_reb[["TEAM_MISSES_opp"]]
X_sorted = X.sort_values("TEAM_MISSES_opp")
predictions_sorted = model.predict(X_sorted)

y = opp_misses_reb["REB"]
model = LinearRegression()
model.fit(X, y)
r2 = model.score(X, y)
print("R2:", r2)
predictions = model.predict(X)
plt.scatter(opp_misses_reb["TEAM_MISSES_opp"], opp_misses_reb["REB"])
plt.plot(X_sorted["TEAM_MISSES_opp"], predictions_sorted, color="red") 

plt.xlim(0, 100)
plt.ylim(0, 100)

plt.xlabel("Opp Team Misses")
plt.ylabel("Rebounds")
plt.title("Opp Team Misses vs Rebounds")

plt.show()

"""Feature: Opponent Team Misses

Correlation: ~0.55
R²: ~0.303

Observation:
Opponent team misses have a moderate positive relationship with
player rebounds.

Basketball reasoning:
More missed shots by the opponent create more opportunities
for players to collect rebounds.

Pregame usable?
No. Actual opponent misses are not known before the game.

Potential transformation:
Use rolling averages of opponent misses from previous games,
such as OPP_MISSES_LAST_5.

Decision:
Candidate feature for future feature engineering."""